In [ ]:
#get the time factor

def get_time_factor(days):
    if days < 0:
        raise ValueError('Day before dep cannot smaller than 0')
    if days <= 7:
        time_factor = 1.35
    elif days <= 21:
        time_factor = 1.1
    else:
        time_factor = 1
    return time_factor

print(get_time_factor(12))
try:
    get_time_factor(-1)
except ValueError:
    print("Invalid value: days must be 0 or greater.")
print(get_time_factor(0))



1.1
ilegal value has been stopped
1.35


In [2]:
class Flight:
    def __init__(self, flight_id, base_fare, seats_remaining, capacity, demand_factor, seasonal_factor, origin, destination):

        #Check case
        if capacity <= 0:
            raise ValueError('Capacity must be greater than 0')
        if not 0 <= seats_remaining <= capacity:
            raise ValueError('Seat remaining must within 0 and Capacity ')
        if base_fare <= 0:
            raise ValueError('Base fare must greater than 0')
        if demand_factor <= 0 or seasonal_factor <= 0:
            raise ValueError('Demand and seasonal factors must be greater than 0')

        # Cast to plain Python types (values may arrive as numpy types from
        # a pandas row, which sqlite3 does not store correctly as INTEGER/REAL).
        self.flight_id = str(flight_id)
        self.base_fare = float(base_fare)
        self.seats_remaining = int(seats_remaining)
        self.capacity = int(capacity)
        self.demand_factor = float(demand_factor)
        self.seasonal_factor = float(seasonal_factor)
        self.origin = origin
        self.destination = destination

    def load_factor(self):
        return 1 - self.seats_remaining/self.capacity

    def update_seats(self, change):
        new_value = self.seats_remaining + change

        if not 0 <= new_value <= self.capacity:
            raise ValueError("seat update is outside 0..capacity")
        else:
            self.seats_remaining = new_value

    def calculate_price(self, days):
        time_factor = get_time_factor(days)
        capacity_factor = 1 + 0.45 * self.load_factor()
        price = self.base_fare * time_factor * capacity_factor * self.demand_factor * self.seasonal_factor
        return round(min(1500,max(45,price)), 2)

    def to_dict(self):
        # Same pattern as Exercise 2: turn a Flight object back into a plain
        # dictionary so it can be written to SQLite or JSON.
        return vars(self).copy()


test_flight = Flight('PA1001', 300, 18, 180, 1.2, 1.0, 'YYZ', 'YVR')
print(test_flight.destination)

YVR


In [3]:
print(test_flight.seats_remaining)
print(test_flight.load_factor())
try:
    test_flight.update_seats(-20)
except ValueError:
    print("Invalid seat update blocked.")

18
0.9
Invalid seat update blocked.


In [4]:
old_price = test_flight.calculate_price(12)
test_flight.update_seats(-5)
new_price = test_flight.calculate_price(12)
print(old_price)
print(new_price)

556.38
561.33


In [5]:
try:
    bad_flight = Flight("PA_TEST", 300, 181, 180, 1.2, 1.0, 'YYZ', 'CHN')
except ValueError:
    print("Invalid flight update blocked.")

Invalid flight update blocked.


## SQLite Persistence

Run these cells in order from this notebook folder. The setup cell rebuilds the demo flights table from CSV, resetting previous database updates. Pricing integration is still pending.

### 1. Load CSV

In [6]:
# Load flight data.
import sqlite3
import pandas as pd

flights = pd.read_csv("flights.csv")


### 2. Create Table and Insert Flights

In [7]:
def setup_flights_table(flights_df):
    """Create the flights table and load one row per validated Flight."""
    conn = sqlite3.connect("potter_airlines.db")
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS flights")
    cursor.execute("""
        CREATE TABLE flights (
            flight_id TEXT PRIMARY KEY,
            origin TEXT NOT NULL,
            destination TEXT NOT NULL,
            departure_date TEXT NOT NULL,
            base_fare REAL NOT NULL,
            capacity INTEGER NOT NULL,
            seats_remaining INTEGER NOT NULL,
            demand_factor REAL NOT NULL,
            seasonal_factor REAL NOT NULL
        )
    """)

    for record in flights_df.to_dict(orient="records"):
        flight = Flight(
            record["flight_id"], record["base_fare"], record["seats_remaining"],
            record["capacity"], record["demand_factor"], record["seasonal_factor"],
            record["origin"], record["destination"]
        )
        flight_dict = flight.to_dict()
        cursor.execute(
            """
            INSERT INTO flights (
                flight_id, origin, destination, departure_date,
                base_fare, capacity, seats_remaining, demand_factor, seasonal_factor
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                flight_dict["flight_id"], flight_dict["origin"],
                flight_dict["destination"], record["departure_date"],
                flight_dict["base_fare"], flight_dict["capacity"],
                flight_dict["seats_remaining"], flight_dict["demand_factor"],
                flight_dict["seasonal_factor"]
            )
        )

    conn.commit()
    conn.close()


setup_flights_table(flights)
print("CSV flights validated through the Flight class and inserted.")

CSV flights validated through the Flight class and inserted.


### 3. Query Flights

In [8]:
def get_flights_by_destination(destination):
    """Return every stored flight row to the given destination."""
    conn = sqlite3.connect("potter_airlines.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT * FROM flights WHERE destination = ?",
        (destination,)
    )
    rows = cursor.fetchall()
    conn.close()
    return rows


destination = "YVR"
rows = get_flights_by_destination(destination)
print("Flights to YVR:", rows)

Flights to YVR: [('PA1001', 'YYZ', 'YVR', '2026-09-17', 300.0, 180, 18, 1.2, 1.0), ('PA1002', 'YYZ', 'YVR', '2026-09-23', 300.0, 180, 90, 1.2, 1.0), ('PA1003', 'YYZ', 'YVR', '2026-10-07', 300.0, 180, 144, 1.2, 1.0), ('PA1004', 'YYZ', 'YVR', '2026-10-08', 300.0, 180, 144, 1.2, 1.0), ('PA1016', 'YYC', 'YVR', '2026-10-17', 180.0, 120, 36, 0.9, 1.0), ('PA1017', 'YYZ', 'YVR', '2026-12-20', 300.0, 180, 90, 1.2, 1.3)]


### 4. Update and Delete a Test Flight

In [9]:
# Demonstrate and verify seat updates and deletion on a test flight.
conn = sqlite3.connect("potter_airlines.db")
cursor = conn.cursor()

cursor.execute(
    "INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
    ("SQL_DEMO", "YYZ", "YVR", "2026-10-01", 300, 180, 18, 1.2, 1.0)
)

cursor.execute(
    "UPDATE flights SET seats_remaining = ? WHERE flight_id = ?",
    (13, "SQL_DEMO")
)
cursor.execute(
    "SELECT flight_id, seats_remaining FROM flights WHERE flight_id = ?",
    ("SQL_DEMO",)
)
updated = cursor.fetchall()
print("Updated test flight:", updated)
assert updated == [("SQL_DEMO", 13)]

cursor.execute("DELETE FROM flights WHERE flight_id = ?", ("SQL_DEMO",))
cursor.execute("SELECT * FROM flights WHERE flight_id = ?", ("SQL_DEMO",))
deleted = cursor.fetchall()
assert deleted == []
print("Test flight deleted:", deleted)

conn.commit()
conn.close()

Updated test flight: [('SQL_DEMO', 13)]
Test flight deleted: []


### 5. Verify Stored Data

In [10]:
# Read persisted flights into a DataFrame and verify the row count.
with sqlite3.connect("potter_airlines.db") as conn:
    stored_flights = pd.read_sql_query("SELECT * FROM flights", conn)
assert len(stored_flights) == len(flights)
print("Stored flight count:", len(stored_flights))
print(stored_flights.head())

Stored flight count: 20
  flight_id origin destination departure_date  base_fare  capacity  \
0    PA1001    YYZ         YVR     2026-09-17      300.0       180   
1    PA1002    YYZ         YVR     2026-09-23      300.0       180   
2    PA1003    YYZ         YVR     2026-10-07      300.0       180   
3    PA1004    YYZ         YVR     2026-10-08      300.0       180   
4    PA1005    YVR         YYZ     2026-09-18      300.0       180   

   seats_remaining  demand_factor  seasonal_factor  
0               18            1.2              1.0  
1               90            1.2              1.0  
2              144            1.2              1.0  
3              144            1.2              1.0  
4                1            1.2              1.0  


## Pricing All Flights (Vectorized)

The SQLite section above loads the flights but does not price them yet. This section:

1. Prepares the departure dates and checks the whole table before calculating anything.
2. Prices every flight at once using Pandas/NumPy (no `for` loop), using the same formula as `Flight.calculate_price`.
3. Filters out sold-out flights and ranks the rest.
4. Updates one flight's seats and shows the price respond, persisting the change to SQLite.
5. Demonstrates one edge case on real data (selling a seat on an already sold-out flight).

`quote_date` is fixed so the notebook gives the same numbers every time it is run.

### 1. Prepare dates and check the table
`departure_date` is stored as text, so it is converted to an actual date first. `days_until_departure` is how many days from `quote_date` until that flight leaves.

The `assert` lines repeat, on the whole table at once, the same assumptions the `Flight` class already checks one flight at a time.

In [11]:
# Fixed "today" for quoting fares, so the notebook is reproducible.
quote_date = pd.Timestamp("2026-09-19")

priced = stored_flights.copy()
priced["departure_date"] = pd.to_datetime(priced["departure_date"])
priced["days_until_departure"] = (priced["departure_date"] - quote_date).dt.days

# Same assumptions as the Flight class, checked on every row at once.
assert priced.notna().all().all()
assert priced["flight_id"].is_unique
assert (priced["capacity"] > 0).all()
assert priced["seats_remaining"].between(0, priced["capacity"]).all()
assert (priced["base_fare"] > 0).all()
assert (priced["demand_factor"] > 0).all()
assert (priced["seasonal_factor"] > 0).all()
print("Input checks passed.")

# Edge case: a flight that has already departed cannot be priced.
departed = priced.loc[priced["days_until_departure"] < 0, "flight_id"].tolist()
print("Already departed, excluded from pricing:", departed)
priced = priced[priced["days_until_departure"] >= 0].copy()

Input checks passed.
Already departed, excluded from pricing: ['PA1001', 'PA1005']


### 2. Price every flight at once
This is the same formula as `Flight.calculate_price`, but applied to the whole column instead of one flight:

```text
price = base_fare x time_factor x capacity_factor x demand_factor x seasonal_factor
```

`np.select` picks the first true condition for `time_factor`, same three tiers as `get_time_factor`.

In [12]:
import numpy as np

days = priced["days_until_departure"]
priced["time_factor"] = np.select(
    [days <= 7, days <= 21],
    [1.35, 1.10],
    default=1.0,
)

priced["load_factor"] = 1 - priced["seats_remaining"] / priced["capacity"]
priced["capacity_factor"] = 1 + 0.45 * priced["load_factor"]

priced["raw_price"] = (
    priced["base_fare"]
    * priced["time_factor"]
    * priced["capacity_factor"]
    * priced["demand_factor"]
    * priced["seasonal_factor"]
)
priced["final_price"] = np.clip(priced["raw_price"], 45, 1500).round(2)

# Robustness check: every computed fare must respect the same bounds as
# Flight.calculate_price, for every flight, not just the ones we inspect.
assert priced["final_price"].between(45, 1500).all()

priced[["flight_id", "origin", "destination", "days_until_departure",
        "seats_remaining", "final_price"]]

,flight_id,origin,destination,days_until_departure,seats_remaining,final_price
1,PA1002,YYZ,YVR,4,90,595.35
2,PA1003,YYZ,YVR,18,144,431.64
3,PA1004,YYZ,YVR,19,144,431.64
5,PA1006,YYZ,YUL,0,60,248.06
6,PA1007,YYZ,YUL,1,0,293.62
7,PA1008,YUL,YYZ,5,120,202.50
8,PA1009,YYZ,YYC,6,16,521.61
9,PA1010,YYC,YYZ,7,80,454.78
10,PA1011,YYZ,YOW,11,85,119.77
11,PA1012,YOW,YYZ,12,20,152.59


In [13]:

check_row = priced.loc[priced["flight_id"] == "PA1002"].iloc[0]
check_flight = Flight(
    check_row["flight_id"], check_row["base_fare"], check_row["seats_remaining"],
    check_row["capacity"], check_row["demand_factor"], check_row["seasonal_factor"],
    check_row["origin"], check_row["destination"],
)
scalar_price = check_flight.calculate_price(int(check_row["days_until_departure"]))
assert abs(scalar_price - check_row["final_price"]) < 0.01
print("Vectorized and single-flight prices agree:", scalar_price)


Vectorized and single-flight prices agree: 595.35


### 3. Filter and rank flights
A sold-out flight (0 seats remaining) cannot be booked, so it is excluded before ranking. The remaining flights are ranked by `final_price`, cheapest first.

In [14]:
available = priced[priced["seats_remaining"] > 0].copy()
print("Sold-out flights excluded:", len(priced) - len(available))

cheapest_available = available.sort_values("final_price").head(5)
cheapest_available[["flight_id", "origin", "destination",
                     "seats_remaining", "final_price"]]


Sold-out flights excluded: 1


,flight_id,origin,destination,seats_remaining,final_price
10,PA1011,YYZ,YOW,85,119.77
19,PA1020,YUL,YYZ,96,147.15
11,PA1012,YOW,YYZ,20,152.59
14,PA1015,YVR,YYC,96,194.24
7,PA1008,YUL,YYZ,120,202.50


### 4. Update an operational value and re-price
Selling seats on flight PA1002 raises its load factor, which raises `capacity_factor` and therefore the price. The new seat count is persisted with a parameterized `UPDATE`, then read back to confirm it saved.

In [15]:
target_id = "PA1002"
target_row = stored_flights.loc[stored_flights["flight_id"] == target_id].iloc[0]
days_to_departure = int(priced.loc[priced["flight_id"] == target_id, "days_until_departure"].iloc[0])

target_flight = Flight(
    target_row["flight_id"], target_row["base_fare"], target_row["seats_remaining"],
    target_row["capacity"], target_row["demand_factor"], target_row["seasonal_factor"],
    target_row["origin"], target_row["destination"],
)

price_before = target_flight.calculate_price(days_to_departure)
target_flight.update_seats(-10)
price_after = target_flight.calculate_price(days_to_departure)

conn = sqlite3.connect("potter_airlines.db")
cursor = conn.cursor()
cursor.execute(
    "UPDATE flights SET seats_remaining = ? WHERE flight_id = ?",
    (target_flight.seats_remaining, target_id),
)
cursor.execute(
    "SELECT seats_remaining FROM flights WHERE flight_id = ?",
    (target_id,),
)
confirmed = cursor.fetchone()
conn.commit()
conn.close()

assert confirmed == (target_flight.seats_remaining,)
print(f"{target_id}: sold 10 seats, {target_row['seats_remaining']} -> "
      f"{target_flight.seats_remaining} seats remaining (confirmed in database).")
print(f"Price moved from {price_before} to {price_after} as the load factor increased.")

PA1002: sold 10 seats, 90 -> 80 seats remaining (confirmed in database).
Price moved from 595.35 to 607.5 as the load factor increased.


### 5. Edge case: selling seats on a sold-out flight
`PA1007` has 0 seats remaining in `flights.csv`. `update_seats` should reject any attempt to sell a seat on it, the same guard already tested on `test_flight` earlier in this notebook, now shown on real data.

In [16]:
soldout_row = stored_flights.loc[stored_flights["flight_id"] == "PA1007"].iloc[0]
soldout_flight = Flight(
    soldout_row["flight_id"], soldout_row["base_fare"], soldout_row["seats_remaining"],
    soldout_row["capacity"], soldout_row["demand_factor"], soldout_row["seasonal_factor"],
    soldout_row["origin"], soldout_row["destination"],
)
print(soldout_flight.flight_id, "seats remaining:", soldout_flight.seats_remaining)

try:
    soldout_flight.update_seats(-1)
except ValueError:
    print("Blocked: cannot sell a seat on a sold-out flight.")


PA1007 seats remaining: 0
Blocked: cannot sell a seat on a sold-out flight.
